In [1]:
# ============================================================
# 라이브러리 Import
# ============================================================

# 데이터 처리 및 분석
import os
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 통계 분석
from scipy import stats
from scipy.stats import shapiro, levene, ttest_ind, chi2_contingency, f_oneway
from scipy.stats import mannwhitneyu, fisher_exact, kruskal
from scipy.stats import skew, kurtosis
from statsmodels.stats.multicomp import pairwise_tukeyhsd, MultiComparison
import pingouin as pg
import scikit_posthocs as sp

# 머신러닝
from sklearn.preprocessing import MinMaxScaler

# 출력 설정
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 참고: seed 고정으로 팀원 간 동일한 결과 재현 가능
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

#모든 컬럼 다 보이게
pd.set_option("display.max_columns", None)
#모든 행도 다 보이게
pd.set_option("display.max_rows", None)


#판다스가 화면에 보여줄 때 잘리지 않게 출력 옵션을 바꾸기
#pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

라이브러리 로드 완료!
한글 폰트 설정 완료!


In [2]:
full_data = pd.read_csv("full_data.csv") #전처리 파일 저장하면 이걸로 시작

In [3]:
# 1. 파일 위치
metadata_file = "./metadata.csv"
data_folder = "./data"

# 2. metadata 불러오기
df = pd.read_csv(metadata_file)

# 3. 컬럼명 공백 제거
df.columns = df.columns.str.strip()

In [18]:
full_data.head()

,Voltage_measured,Current_measured,Temperature_measured,Current_load,Voltage_load,Time,filename,type,battery_id,start_time,Sense_current,Battery_current,Current_ratio,Battery_impedance,Rectified_Impedance,Current_charge,Voltage_charge
0,4.246711,0.000252,6.212696,0.0002,0.000,0.000,00001.csv,discharge,B0047,[2010. 7. 21. 15. 0. ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4.246764,-0.001411,6.234019,0.0002,4.262,9.360,00001.csv,discharge,B0047,[2010. 7. 21. 15. 0. ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4.039277,-0.995093,6.250255,1.0000,3.465,23.281,00001.csv,discharge,B0047,[2010. 7. 21. 15. 0. ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4.019506,-0.996731,6.302176,1.0000,3.451,36.406,00001.csv,discharge,B0047,[2010. 7. 21. 15. 0. ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4.004763,-0.992845,6.361645,1.0000,3.438,49.625,00001.csv,discharge,B0047,[2010. 7. 21. 15. 0. ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
type_count = full_data["type"].value_counts(dropna=False)
print("\n[타입별 개수]")
print(type_count)


[타입별 개수]
type
charge       6512876
discharge     770070
impedance      93888
Name: count, dtype: int64


In [24]:
battery_count = full_data["battery_id"].value_counts(dropna=False)
print("\n[battery_id별 행 개수 상위 20개]")
print(battery_count.head(20))

print("\n[battery_id 개수]")
print(full_data["battery_id"].nunique())


[battery_id별 행 개수 상위 20개]
battery_id
B0006    604802
B0005    604802
B0007    604802
B0034    411228
B0033    411228
B0036    411228
B0018    317220
B0043    279536
B0042    279536
B0044    279536
B0054    261592
B0056    261589
B0055    261589
B0039    139767
B0040    139767
B0038    139767
B0041    139686
B0047    127404
B0045    127404
B0048    127404
Name: count, dtype: int64

[battery_id 개수]
34


In [25]:
type_battery_count = full_data.groupby("type")["battery_id"].nunique()
print("\n[타입별 battery_id 개수]")
print(type_battery_count)


[타입별 battery_id 개수]
type
charge       34
discharge    34
impedance    34
Name: battery_id, dtype: int64


In [ ]:
fill_ratio = full_data.notnull().groupby(full_data["type"]).mean() * 100
print("타입별 notnull 비율(%)")
print(fill_ratio.T)

타입별 컬럼 채워진 비율(%)
type                      charge  discharge  impedance
Voltage_measured       99.996054      100.0       0.00
Current_measured       99.996054      100.0       0.00
Temperature_measured   99.996054      100.0       0.00
Current_load            0.000000      100.0       0.00
Voltage_load            0.000000      100.0       0.00
Time                  100.000000      100.0       0.00
filename              100.000000      100.0     100.00
type                  100.000000      100.0     100.00
battery_id            100.000000      100.0     100.00
start_time            100.000000      100.0     100.00
Sense_current           0.000000        0.0     100.00
Battery_current         0.000000        0.0     100.00
Current_ratio           0.000000        0.0     100.00
Battery_impedance       0.000000        0.0     100.00
Rectified_Impedance     0.000000        0.0      81.25
Current_charge        100.000000        0.0       0.00
Voltage_charge        100.000000        0.0     

In [4]:
def clear_time(x):
    # 사용할 리스트
    nums = []

    # 임시 저장용
    current = ''

    # 반복문으로 숫자만 뽑아내기
    for ch in x:
        # 숫자형이거나 형태 다른 숫자 뽑아내기
        if ch.isdigit() or ch in ['.', '_', 'e', 'E', '+']:
            current += ch
        else:
            if current != '':
                nums.append(current)
                current = ''

    # 숫자 남아있을 수도 있으니 한번 더 리스트에 append
    if current != '':
        nums.append(current)

    # 연월일시분초 하면 최소 6자리, 그 이하는 버림
    if len(nums) < 6:
        return None

    # num 리스트 숫자형으로 변환, 연월일시분초이므로 앞에 6개만 숫자 변환
    try:
        nums = list(map(float, nums[:6]))
    except:
        return None

    # 각 숫자에 의미 부여
    year, month, day, hour, minute, second = nums

    # 이상치 제거
    if not (
        2000 < year
        and 1 <= month <= 12
        and 1 <= day <= 31
        and 0 <= hour < 24
        and 0 <= minute < 60
        and 0 <= second < 60
    ):
        return None

    # Timestamp로 date_time 변환
    return pd.Timestamp(
        int(year), int(month), int(day),
        int(hour), int(minute), int(second)
    )


df['start_time'] = df['start_time'].apply(clear_time)
df = df.dropna(subset=['start_time'])

In [5]:
df.nunique()

type                      3
start_time             2443
ambient_temperature       5
battery_id               34
test_id                 615
uid                    7412
filename               7412
Capacity               2703
Re                     1910
Rct                    1910
dtype: int64

In [ ]:
#for i in range(1, 10):
    #df1 = pd.read_csv(f'data/data1/0000{i}.csv')
    #print(i, df1.columns)

1 Index(['Voltage_measured', 'Current_measured', 'Temperature_measured', 'Current_load', 'Voltage_load', 'Time'], dtype='str')
2 Index(['Sense_current', 'Battery_current', 'Current_ratio', 'Battery_impedance', 'Rectified_Impedance'], dtype='str')
3 Index(['Voltage_measured', 'Current_measured', 'Temperature_measured', 'Current_charge', 'Voltage_charge', 'Time'], dtype='str')
4 Index(['Sense_current', 'Battery_current', 'Current_ratio', 'Battery_impedance', 'Rectified_Impedance'], dtype='str')
5 Index(['Voltage_measured', 'Current_measured', 'Temperature_measured', 'Current_load', 'Voltage_load', 'Time'], dtype='str')
6 Index(['Voltage_measured', 'Current_measured', 'Temperature_measured', 'Current_charge', 'Voltage_charge', 'Time'], dtype='str')
7 Index(['Voltage_measured', 'Current_measured', 'Temperature_measured', 'Current_load', 'Voltage_load', 'Time'], dtype='str')
8 Index(['Voltage_measured', 'Current_measured', 'Temperature_measured', 'Current_charge', 'Voltage_charge', 'Time'],

1번 5번 세트 /3번 'Voltage_charge' / 2번 4번 세트

- 00001.csv → Discharge(방전) = 첫 번째 방전 기록
- 00002.csv → Impedance(임피던스) = 그 사이 상태 진단
- 00003.csv → Charge(충전) = 다음 충전 기록
- 00004.csv → Impedance(임피던스) = 또 상태 진단
- 00005.csv → Discharge(방전) = 다음 방전 기록

데이터는 한 줄 한 줄이 “일반 거래 데이터”가 아니라, 배터리의 충전/방전/임피던스 측정 사이클을 의미합니다.
=> 리튬이온 배터리를 여러 번 충전·방전·임피던스 측정한 실험 기록

Charge → Discharge = 1 cycle

- type	실험 유형 (charge, discharge, impedance)
- start_time	테스트 시작 시간
- ambient_temperature	실험 환경의 온도(°C)
- battery_id	배터리 식별 ID
- test_id	실험 ID
- uid	실험 샘플 ID
- filename	데이터 파일명
- Capacity	배터리 현재 용량(Ah)
- Re	전해질 저항(Ω)
- Rct	충전 전달 저항(Ω)
- Voltage_measured	배터리 측정 전압(V)
- Current_measured	배터리 측정 전류(A)
- Temperature_measured	배터리 실시간 온도(°C)
- Current_load	배터리 부하 전류(A)
- Voltage_load	배터리 부하 전압(V)

In [6]:
df['battery_id'].nunique()

34

In [7]:
df['battery_id'].value_counts()

battery_id
B0006    607
B0005    607
B0007    607
B0034    477
B0033    477
B0036    477
B0018    312
B0043    268
B0042    268
B0044    268
B0054    246
B0056    245
B0055    245
B0047    182
B0045    182
B0048    182
B0046    182
B0041    160
B0053    133
B0039    117
B0040    117
B0038    117
B0032     96
B0029     96
B0030     96
B0031     96
B0028     77
B0027     77
B0025     77
B0026     77
B0049     61
B0050     61
B0052     61
B0051     61
Name: count, dtype: int64

In [8]:
summary = pd.DataFrame({
    "dtype": df.dtypes,
    "non_null": df.count(),
    "null": df.isnull().sum()
})

summary

,dtype,non_null,null
type,str,7412,0
start_time,datetime64[us],7412,0
ambient_temperature,int64,7412,0
battery_id,str,7412,0
test_id,int64,7412,0
uid,int64,7412,0
filename,str,7412,0
Capacity,str,2745,4667
Re,str,1910,5502
Rct,str,1910,5502


# []처리

In [12]:
df['Capacity'].isnull().sum()

np.int64(4667)

In [9]:
df['Capacity'].value_counts().sort_values(ascending=False)

Capacity
[]                      25
0                       19
1.6743047446975208       1
1.5243662105099023       1
1.5080762969973425       1
1.4835577960067696       1
1.4671391666146525       1
1.448858156982267        1
1.4458534180949325       1
1.431118266178283        1
1.4192745516578533       1
1.3999974221808271       1
1.3885156686114437       1
1.3652234669528964       1
1.4060442272970963       1
1.4057541613525781       1
1.3867662881727878       1
1.3705053061420223       1
1.3497431114823601       1
1.3251533367939783       1
1.3111943869635805       1
1.3394234405932892       1
1.2849157142560723       1
1.281719159327807        1
1.2600051642942829       1
1.266070385993126        1
1.2413590294280803       1
1.2298873703485311       1
1.228088614506696        1
1.2139857189070042       1
1.2173435064564189       1
1.2073445944515917       1
1.18804049227045         1
1.186304749886542        1
1.2613935471028506       1
1.2648246158850653       1
1.2466494548982199 

In [10]:
df["Capacity"].apply(type).value_counts()

Capacity
<class 'float'>    4667
<class 'str'>      2745
Name: count, dtype: int64

In [9]:
df.loc[df["Capacity"].map(type) == str, "Capacity"]

0         1.6743047446975208
4         1.5243662105099023
6         1.5080762969973425
8         1.4835577960067696
10        1.4671391666146525
12         1.448858156982267
16        1.4458534180949325
20         1.431118266178283
22        1.4192745516578533
24        1.3999974221808271
26        1.3885156686114437
28        1.3652234669528964
32        1.4060442272970963
36        1.4057541613525781
38        1.3867662881727878
40        1.3705053061420223
42        1.3497431114823601
44        1.3251533367939783
48        1.3111943869635805
50                         0
52        1.3394234405932892
54        1.2849157142560723
56         1.281719159327807
60        1.2600051642942829
62         1.266070385993126
64        1.2413590294280803
66        1.2298873703485311
68         1.228088614506696
72        1.2139857189070042
74        1.2173435064564189
76        1.2073445944515917
78          1.18804049227045
80         1.186304749886542
84        1.2613935471028506
88        1.26

In [14]:
cap_text = df["Capacity"].astype("string").str.strip()

df.loc[cap_text == "[]", "Capacity"] = pd.NA
df["Capacity"] = pd.to_numeric(df["Capacity"], errors="coerce")

In [15]:
summary = pd.DataFrame({
    "dtype": df.dtypes,
    "non_null": df.count(),
    "null": df.isnull().sum()
})

summary

,dtype,non_null,null
type,str,7412,0
start_time,datetime64[us],7412,0
ambient_temperature,int64,7412,0
battery_id,str,7412,0
test_id,int64,7412,0
uid,int64,7412,0
filename,str,7412,0
Capacity,float64,2720,4692
Re,str,1910,5502
Rct,str,1910,5502
